# Modified Attention-PestNet with Channel-Reduced HSDPA (CR-HSDPA)
**Thesis Topic:** A Modified Attention-PestNet with Channel-Reduced Hierarchical Scaled Dot-Product Attention for Efficient Insect Pest Detection  
**Authors:** Lean Adrian Murillo, James Oliver C. Mendoza, DM Rashid P. Ferrer, Charisse P. Barbosa (University of Mindanao)  
**Target Architecture:** CR-HSDPA (+CR-0.25 Aggressive Channel Reduction, r=0.25)  
---
This notebook sets up the environment, prepares the IP102 benchmark dataset, verifies model parameter reduction, and trains the **+CR-0.25** ablation variant on Kaggle GPU.

In [ ]:
# 1. Setup repository and install ultralytics in editable mode
import os
os.environ["WANDB_MODE"] = "disabled"
import sys
import shutil

REPO_DIR = "/kaggle/working/HSDPA"
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/90377Sednaaa/HSDPA.git {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git pull

%cd {REPO_DIR}
!pip install -q -e .

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print("Environment successfully set up at:", REPO_DIR)


In [ ]:
# 2. Check GPU environment and CUDA capabilities
import torch
from ultralytics import YOLO

print("PyTorch version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
device_count = torch.cuda.device_count()
print("GPU Count:", device_count)
for i in range(device_count):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")


In [ ]:
# 3. Verify IP102 dataset directory and images
import os
import yaml

DATA_ROOT = "/kaggle/input/ip102-yolov5/IP102_YOLOv5"
if not os.path.exists(DATA_ROOT):
    for root, dirs, files in os.walk("/kaggle/input"):
        if "images" in dirs:
            DATA_ROOT = root
            break

print(f"Dataset root verified at: {DATA_ROOT}")
train_count = len(os.listdir(os.path.join(DATA_ROOT, "images/train")))
val_count = len(os.listdir(os.path.join(DATA_ROOT, "images/val")))
print(f"Train images: {train_count}")
print(f"Validation images: {val_count}")

# Update path in cfg/ip102.yaml dynamically
with open("cfg/ip102.yaml", "r") as f:
    ip102_cfg = yaml.safe_load(f)
ip102_cfg["path"] = DATA_ROOT
with open("cfg/ip102.yaml", "w") as f:
    yaml.dump(ip102_cfg, f, default_flow_style=False, sort_keys=False)
print("cfg/ip102.yaml ready!")


In [ ]:
# 4. Compare Model Complexity: Baseline HSDPA vs. Proposed CR-HSDPA (+CR-0.25)
from ultralytics import YOLO

print("=" * 65)
print("BASELINE MODEL (Attention-PestNet / HSDPA):")
baseline_model = YOLO("cfg/hsdpa.yaml")
baseline_model.info(detailed=False)

print("=" * 65)
print("PROPOSED MODEL (+CR-0.25, r=0.25):")
cr025_model = YOLO("cfg/cr_hsdpa_025.yaml")
cr025_model.info(detailed=False)

p_base = sum(p.numel() for p in baseline_model.model.parameters())
p_prop = sum(p.numel() for p in cr025_model.model.parameters())
diff = p_base - p_prop
pct = (diff / p_base) * 100
print(f"Parameter Reduction: -{diff:,} params (-{pct:.2f}%)")
print("=" * 65)


In [ ]:
# 5. Train Model (+CR-0.25 on IP102)
import torch
import os, glob
os.environ["WANDB_MODE"] = "disabled"
from ultralytics import YOLO

MODEL_CFG = "cfg/cr_hsdpa_025.yaml"
PROJECT_NAME = "CR_HSDPA"
EXP_NAME = "ip102_cr025"

# Check for existing checkpoint to resume seamlessly
resume_ckpt = None
candidate_ckpts = glob.glob(f"/kaggle/input/**/{EXP_NAME}/weights/last.pt", recursive=True) + \
                  glob.glob(f"{PROJECT_NAME}/{EXP_NAME}/weights/last.pt")

if candidate_ckpts:
    resume_ckpt = candidate_ckpts[0]
    print(f"Found checkpoint to resume: {resume_ckpt}")
    model = YOLO(resume_ckpt)
    resume_flag = True
else:
    print(f"Initializing fresh model from {MODEL_CFG}")
    model = YOLO(MODEL_CFG)
    resume_flag = False

# Use multi-GPU if dual T4 is active, else single GPU
device_arg = [0, 1] if torch.cuda.device_count() >= 2 else 0

# Train for Stage 1 (50 epochs) with baseline batch size 32
results = model.train(
    data="cfg/ip102.yaml",
    epochs=50,                # Stage 1 target (safe < 6 hours execution)
    batch=32,                 # Matching baseline study effective batch size
    imgsz=640,
    device=device_arg,
    optimizer="SGD",          # Matching proposal SGD setup
    lr0=0.01,
    momentum=0.9,
    amp=True,                 # Mixed precision on T4
    save_period=10,           # Save every 10 epochs
    project=PROJECT_NAME,
    name=EXP_NAME,
    exist_ok=True,
    workers=4,                # Fully utilize 4 vCPUs
    resume=resume_flag,
    verbose=True
)


In [ ]:
# 6. Verify and summarize output files
import os, glob

print("Generated weight checkpoints and results:")
for f in glob.glob(f"{PROJECT_NAME}/**/weights/*.pt", recursive=True):
    size_mb = os.path.getsize(f) / (1024 * 1024)
    print(f"  {f} ({size_mb:.2f} MB)")
